# Why Attention Matters
### The idea that taught machines to focus — and changed AI forever

---

## 📖 The Story

It is 2014. Neural machine translation exists, but it is broken for long sentences. The culprit: the entire input sentence is compressed into a **single fixed-size vector** before decoding begins. The longer the sentence, the more information is lost in that compression.

Dzmitri Bahdanau asks: *"What if the decoder could look back at every word in the input and decide what to focus on — at each step?"*

This notebook builds that idea from scratch. By the end, you will understand exactly how attention works mathematically, implement it in NumPy, verify it with PyTorch, and visualize it on real text — the same way researchers first saw it working in 2015.

---

**What this notebook covers:**
- The RNN bottleneck problem — demonstrated with code
- Dot-product attention from scratch using NumPy
- Attention weight visualization as a heatmap on real words
- PyTorch verification of our scratch implementation
- Experiments showing how attention changes with sequence length

**Prerequisites:**
- Basic understanding of neural networks (forward pass)
- NumPy fundamentals (matrix multiplication, softmax)
- Python basics

**Note:** This topic uses synthetic text sequences built in-notebook — no external dataset download needed. All data is constructed to clearly illustrate the concept.

In [ ]:
# --- All Imports ---
import numpy as np                        # Core math — attention from scratch
import matplotlib.pyplot as plt           # Plotting attention heatmaps
import seaborn as sns                     # Beautiful heatmap styling
import torch                              # PyTorch — for library verification
import torch.nn as nn                     # PyTorch neural network modules
import torch.nn.functional as F          # Softmax and other functions
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)                        # Reproducibility
torch.manual_seed(42)                     # PyTorch reproducibility

print("PyTorch version:", torch.__version__)
print("All imports successful ✅")

## Part 1: Theory Recap

Five things to know before we write a single line of code:

- **RNNs have a bottleneck**: The encoder compresses the entire input sequence into ONE fixed-size vector. Long sequences lose information — this is why early neural translation was bad at long sentences.
- **Attention fixes this with a weighted lookup**: Instead of one vector, the decoder gets access to ALL encoder hidden states and learns which ones to focus on at each output step.
- **Attention weights are a probability distribution**: softmax(scores) → weights that sum to 1. High weight = high focus on that input position.
- **The context vector is a weighted average**: cₜ = Σᵢ αₜᵢ · hᵢ — a dynamic, step-specific summary of the input tailored to what the decoder needs next.
- **Attention is interpretable**: The weight matrix α can be visualized as a heatmap — you can literally see what the model is focusing on when generating each output word.

## Demonstrating the RNN Bottleneck Problem

Before building attention, let us first see exactly WHY the old approach fails.

We simulate an RNN encoder that processes a sentence and produces a single context vector. Then we show how information from early tokens gets diluted as sequence length increases.

**Real-world parallel:** This is exactly what happened with early Google Translate. A 5-word sentence translated well. A 50-word sentence was often garbled.

In [ ]:
def simple_rnn_step(x, h_prev, W_h, W_x, b):
    """
    One step of a simple RNN.
    h_t = tanh(W_h @ h_{t-1} + W_x @ x_t + b)
    INTERVIEW NOTE: Each step overwrites the hidden state.
    Early token information gets diluted with each new step.
    """
    return np.tanh(W_h @ h_prev + W_x @ x + b)

# Setup: small RNN to illustrate the bottleneck
hidden_size = 8
embed_size  = 4

# Random but fixed weights (simulating a trained RNN)
W_h = np.random.randn(hidden_size, hidden_size) * 0.1
W_x = np.random.randn(hidden_size, embed_size)  * 0.1
b   = np.zeros(hidden_size)

# Simulate encoding sentences of increasing length
# Each token is a random embedding (simulating word vectors)
sequence_lengths = [3, 5, 10, 20, 50]
first_token_similarities = []

print("Demonstrating how RNN loses early token information:\n")
print(f"{'Seq Length':>12} | {'Similarity of final state to first token':>42}")
print("-" * 58)

for seq_len in sequence_lengths:
    # Create token embeddings
    tokens = np.random.randn(seq_len, embed_size)

    # Run RNN — record hidden state after first token
    h = np.zeros(hidden_size)
    h_after_first = None

    for t, token in enumerate(tokens):
        h = simple_rnn_step(token, h, W_h, W_x, b)
        if t == 0:
            h_after_first = h.copy()  # Save state right after first token

    h_final = h  # Final hidden state = the bottleneck vector

    # Cosine similarity between final state and state after first token
    # High similarity = first token info preserved; Low = lost
    cos_sim = np.dot(h_final, h_after_first) / (
        np.linalg.norm(h_final) * np.linalg.norm(h_after_first) + 1e-8
    )
    first_token_similarities.append(cos_sim)
    print(f"{seq_len:>12} | {cos_sim:>42.4f}")

print("\n→ As sequence gets longer, the final hidden state looks LESS like")
print("  the state after the first token. Early information is being lost.")
print("  This is the bottleneck problem attention was designed to solve.")

## Part 2: Attention From Scratch

Now we build the attention mechanism from scratch using only NumPy.

**The three steps of attention:**
1. Compute **scores** — how relevant is each encoder state to the current decoder state?
2. Apply **softmax** — convert scores to a probability distribution (weights that sum to 1)
3. Compute **context vector** — weighted sum of encoder states using those weights

**Real example:** We use the sentence `["The", "cat", "sat", "on", "mat"]` to make the output human-readable.

In [ ]:
def softmax(x):
    """
    Numerically stable softmax.
    Subtract max before exp to prevent overflow.
    INTERVIEW NOTE: Always use stable softmax in practice.
    softmax(x)ᵢ = exp(xᵢ) / Σⱼ exp(xⱼ)
    """
    e_x = np.exp(x - np.max(x))  # stability trick
    return e_x / e_x.sum()

def dot_product_attention(query, keys, values):
    """
    Dot-product attention from scratch.

    Args:
        query  : decoder state at current step — shape (d,)
        keys   : all encoder hidden states — shape (seq_len, d)
        values : all encoder hidden states — shape (seq_len, d)
                 (in basic attention, keys == values == encoder states)

    Returns:
        context : weighted sum of values — shape (d,)
        weights : attention weights — shape (seq_len,) — sums to 1

    INTERVIEW NOTE: Keys and Values are the same in basic attention.
    In transformer self-attention, Q, K, V are learned projections.
    """
    # Step 1: Compute raw scores — dot product of query with each key
    # score(query, keyᵢ) = query · keyᵢ
    # Higher score = more relevant encoder state
    scores = keys @ query   # shape: (seq_len,)

    print(f"  Step 1 — Raw scores (query · each key): {np.round(scores, 3)}")

    # Step 2: Apply softmax to get attention weights
    # Converts scores into a probability distribution
    weights = softmax(scores)  # shape: (seq_len,) — sums to 1.0

    print(f"  Step 2 — Attention weights (softmax):   {np.round(weights, 3)}")
    print(f"           Weights sum to: {weights.sum():.4f} ✅" if abs(weights.sum() - 1.0) < 1e-6 else "  ❌ Weights don't sum to 1")

    # Step 3: Compute context vector — weighted sum of values
    # cₜ = Σᵢ αᵢ · valueᵢ
    context = weights @ values   # shape: (d,)

    print(f"  Step 3 — Context vector (weighted sum): shape {context.shape}")

    return context, weights

print("Attention functions defined ✅")

In [ ]:
# Real example: translating "The cat sat on mat"
# We simulate encoder hidden states for each input word

input_words  = ["The", "cat", "sat", "on", "mat"]
output_words = ["Le",  "chat", "s'est", "assis", "sur", "le", "tapis"]

seq_len   = len(input_words)
embed_dim = 6   # Small embedding size for clarity

# Simulate encoder hidden states — one per input word
# In a real model these come from the RNN encoder
# We make 'cat' distinctive so we can verify attention focuses on it
encoder_states = np.random.randn(seq_len, embed_dim)
# Make 'cat' (index 1) have a strong unique signal in dim 0
encoder_states[1, 0] = 5.0

# Simulate decoder query for generating "chat" (French for cat)
# We make it align strongly with the 'cat' encoder state
decoder_query = np.zeros(embed_dim)
decoder_query[0] = 4.0   # Strong signal in same dim as 'cat'

print("=" * 55)
print("Translating: 'The cat sat on mat' → French")
print("Decoder is generating: 'chat' (French for 'cat')")
print("=" * 55)
print()

context, attn_weights = dot_product_attention(
    query=decoder_query,
    keys=encoder_states,
    values=encoder_states
)

print()
print("=" * 55)
print("Attention weights per input word:")
for word, weight in zip(input_words, attn_weights):
    bar = '█' * int(weight * 40)
    print(f"  {word:6s}: {weight:.4f}  {bar}")
print()
print(f"→ Model correctly focuses most on '{input_words[np.argmax(attn_weights)]}'")
print(f"  when generating the French word 'chat' (cat) ✅")

## What Just Happened?

We just built and ran a complete attention mechanism from scratch.

- The encoder produced one hidden state per input word — 5 states for 5 words
- The decoder had a query representing what it was about to generate ("chat" = cat)
- Attention computed how similar the query was to each encoder state (dot products)
- Softmax turned those similarities into weights summing to 1
- The model correctly assigned the highest weight to the word "cat"
- The context vector it received was dominated by the "cat" encoder state

**This is the entire idea of attention.** The math that powers GPT, BERT, and Claude is built on exactly this operation — scaled up and extended with multiple heads.

In [ ]:
# --- Visualisation 1: Attention Weight Heatmap ---
# Compute attention weights for EVERY output word against EVERY input word
# This creates the full attention matrix — the classic NLP visualization

# Simulate decoder queries for each output word
# In a real model, these come from the RNN decoder at each step
decoder_queries = np.random.randn(len(output_words), embed_dim)

# Make each output word attend roughly to its aligned input word
# Le→The(0), chat→cat(1), s'est+assis→sat(2), sur→on(3), le+tapis→mat(4)
alignments = [0, 1, 2, 2, 3, 4, 4]  # which input word each output word aligns to
for i, align_idx in enumerate(alignments):
    decoder_queries[i, 0] = encoder_states[align_idx, 0] * 0.9

# Build full attention matrix
attention_matrix = np.zeros((len(output_words), len(input_words)))
for i, query in enumerate(decoder_queries):
    scores  = encoder_states @ query
    weights = softmax(scores)
    attention_matrix[i] = weights

# Plot the heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    attention_matrix,
    xticklabels=input_words,
    yticklabels=output_words,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    linewidths=0.5,
    ax=ax
)
ax.set_title('Attention Weight Matrix\n"The cat sat on mat" → French Translation',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Input Words (English)', fontsize=11)
ax.set_ylabel('Output Words (French)', fontsize=11)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
plt.tight_layout()
plt.show()

print("Each row shows: when generating this output word, how much did the model")
print("focus on each input word? Darker = more attention.")
print("\nThis is the famous attention heatmap — it makes the model interpretable.")

In [ ]:
# --- Visualisation 2: RNN information loss vs Attention ---
# Directly compare how much early-token info is preserved

seq_lengths = list(range(2, 30))
rnn_similarities = []

for seq_len in seq_lengths:
    tokens = np.random.randn(seq_len, embed_size)
    h = np.zeros(hidden_size)
    h_after_first = None

    for t, token in enumerate(tokens):
        h = simple_rnn_step(token, h, W_h, W_x, b)
        if t == 0:
            h_after_first = h.copy()

    cos_sim = np.dot(h, h_after_first) / (
        np.linalg.norm(h) * np.linalg.norm(h_after_first) + 1e-8
    )
    rnn_similarities.append(cos_sim)

# Attention always has direct access to all positions — similarity is always 1.0
attention_access = [1.0] * len(seq_lengths)  # Attention can always directly access position 0

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(seq_lengths, rnn_similarities, color='red', linewidth=2.5,
        marker='o', markersize=4, label='RNN: similarity of final state to first token')
ax.axhline(y=1.0, color='green', linewidth=2.5, linestyle='--',
           label='Attention: direct access to all positions (always 1.0)')
ax.fill_between(seq_lengths, rnn_similarities, 1.0, alpha=0.15, color='red',
                label='Information lost by RNN')
ax.set_title('RNN Information Loss vs Attention Direct Access\nWhy Attention Matters',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Sequence Length (number of tokens)', fontsize=11)
ax.set_ylabel('Similarity to First Token Information', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.15)
plt.tight_layout()
plt.show()

print("Red line: As sequence grows, RNN final state loses information about early tokens.")
print("Green line: Attention always has direct access to every position — no loss.")

## Part 3: PyTorch Verification

Now we verify our scratch implementation against PyTorch's `nn.MultiheadAttention`.

We use `num_heads=1` and disable the learned projections (`_qkv_same_embed_dim`) to match our simple dot-product attention.

**Important:** PyTorch's MultiheadAttention includes learned linear projections (W_Q, W_K, W_V) which our scratch version does not. So instead of comparing PyTorch's module directly, we implement the scaled dot-product attention formula using `F.scaled_dot_product_attention` — the core operation — and verify against our scratch output.

In [ ]:
# Verify our attention implementation using PyTorch's core function
# F.scaled_dot_product_attention computes: softmax(QKᵀ/√d) · V
# We set scale=1.0 to match our unscaled dot-product attention

# Convert our NumPy arrays to PyTorch tensors
# Shape required: (batch, seq_len, dim)
q_torch = torch.tensor(decoder_query, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1, 1, d)
k_torch = torch.tensor(encoder_states, dtype=torch.float32).unsqueeze(0)              # (1, seq, d)
v_torch = torch.tensor(encoder_states, dtype=torch.float32).unsqueeze(0)              # (1, seq, d)

# PyTorch scaled dot-product attention
# scale_factor controls the 1/√d scaling — we set it manually
# Default PyTorch uses 1/√d; we use 1.0 to match our scratch version
with torch.no_grad():
    # Compute raw scores manually to match our unscaled version
    scores_torch = torch.matmul(q_torch, k_torch.transpose(-2, -1))  # (1, 1, seq)
    weights_torch = F.softmax(scores_torch, dim=-1)                   # (1, 1, seq)
    context_torch = torch.matmul(weights_torch, v_torch)              # (1, 1, d)

weights_torch_np = weights_torch.squeeze().numpy()  # (seq,)
context_torch_np = context_torch.squeeze().numpy()  # (d,)

print("=== Scratch vs PyTorch Verification ===")
print()
print("Attention Weights:")
print(f"  Scratch : {np.round(attn_weights, 6)}")
print(f"  PyTorch : {np.round(weights_torch_np, 6)}")
max_diff_weights = np.max(np.abs(attn_weights - weights_torch_np))
print(f"  Max difference: {max_diff_weights:.2e} {'✅ Match' if max_diff_weights < 1e-5 else '❌ Mismatch'}")

print()
print("Context Vector (first 3 dims):")
print(f"  Scratch : {np.round(context[:3], 6)}")
print(f"  PyTorch : {np.round(context_torch_np[:3], 6)}")
max_diff_context = np.max(np.abs(context - context_torch_np))
print(f"  Max difference: {max_diff_context:.2e} {'✅ Match' if max_diff_context < 1e-5 else '❌ Mismatch'}")

## Part 4: Hyperparameter Experiments

Two important experiments:

1. **Effect of score magnitude on attention sharpness** — when scores are large, softmax becomes more peaked (one winner takes almost all). When scores are small, attention spreads more evenly. This is why the Transformer uses **scaled dot-product attention** (divides by √d) — to prevent softmax from becoming too peaked in high dimensions.

2. **Effect of sequence length on context vector quality** — does the context vector remain useful as the sequence grows longer?

In [ ]:
# Experiment 1: Score scaling effect on attention sharpness
# This directly motivates the √d scaling in transformers

base_scores = np.array([1.0, 0.5, 0.2, 0.1, 0.05])  # 5 input positions
scale_factors = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
words = ["cat", "sat", "on", "the", "mat"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, scale in enumerate(scale_factors):
    scaled_scores = base_scores * scale
    weights = softmax(scaled_scores)
    entropy = -np.sum(weights * np.log(weights + 1e-9))  # Measure of spread

    axes[i].bar(words, weights, color='steelblue', edgecolor='black')
    axes[i].set_title(f'Scale = {scale}x\nEntropy = {entropy:.2f} (lower = sharper)',
                      fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Attention Weight', fontsize=9)
    axes[i].set_ylim(0, 1.05)
    axes[i].grid(True, alpha=0.3, axis='y')
    for j, (word, w) in enumerate(zip(words, weights)):
        axes[i].text(j, w + 0.02, f'{w:.2f}', ha='center', fontsize=8)

plt.suptitle('Effect of Score Magnitude on Attention Sharpness\n'
             'This is why Transformers use Scaled Dot-Product Attention (÷√d)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key insight: Large scores → very peaked (sharp) attention → model ignores most tokens")
print("Small scores → very flat attention → model treats all tokens equally (no focus)")
print("Dividing by √d keeps scores in a range where softmax is informative but not extreme")

## Common Mistakes Students Make

Three things that trip up almost everyone when first learning attention:

**Mistake 1: Confusing attention weights with importance scores**
Attention weights tell you what the model focused on — not which words are objectively most important. A model might focus heavily on a function word like "not" because it changes the meaning of everything around it.

**Mistake 2: Thinking attention replaces the encoder**
In the original Bahdanau attention, the encoder is still an RNN. Attention is added ON TOP of the RNN to give the decoder access to all hidden states — not a replacement. The Transformer (next topics) is the architecture that removes the RNN entirely.

**Mistake 3: Thinking softmax always produces sharp attention**
As we showed in the experiment above, softmax sharpness depends entirely on the magnitude of the input scores. With small scores, attention spreads evenly across all tokens. This is why scaling matters — and why the Transformer scales by 1/√d before the softmax.

## Part 5: Interview Corner

**Question: Why do transformers outperform RNNs? Explain from first principles.**

This directly addresses the module learning outcome. Here is the complete FAANG-level answer:

**1. The bottleneck problem:**
RNN seq2seq models compress the entire input into one fixed-size vector. Long sequences lose early-token information because each RNN step overwrites the hidden state. Attention gives the decoder direct access to ALL encoder states — no compression, no loss.

**2. Parallelization:**
RNNs are inherently sequential — hidden state at step t requires step t-1. You cannot parallelize across time steps. Transformer attention computes all pairwise relationships simultaneously — training is dramatically faster on modern GPUs.

**3. Long-range dependencies:**
In an RNN, the gradient signal from token 1 must travel through every intermediate step to influence token 50 — it often vanishes (vanishing gradient problem). In attention, token 1 and token 50 interact directly in one step — gradient flows freely.

**4. Scalability:**
Transformers scale predictably with more data and more parameters (scaling laws — covered in Topic 7). RNNs do not exhibit the same scaling behavior.

**The one-sentence answer:**
Transformers outperform RNNs because attention gives every token direct access to every other token in one parallel operation — eliminating the sequential bottleneck, the vanishing gradient problem, and the fixed-size information compression that limited RNNs.

## Key Takeaways

Five bullets you must remember for placement interviews:

- **RNNs fail on long sequences due to the bottleneck problem**: The entire input is compressed into one fixed-size vector. Early-token information is diluted with each new RNN step. Attention was invented specifically to fix this.
- **Attention = learned weighted average of all encoder states**: At each decoding step, attention scores each encoder state, applies softmax, and computes a context vector = Σᵢ αᵢ · hᵢ. The decoder gets a fresh, focused summary at every step.
- **Attention weights are interpretable**: The weight matrix α can be visualized as a heatmap showing which input words the model focused on when generating each output word. This is a major advantage over black-box RNNs.
- **Score magnitude controls attention sharpness**: Large scores → peaked softmax → model focuses on one token. Small scores → flat softmax → model spreads attention evenly. Transformers divide by √d to prevent scores from becoming too large in high dimensions.
- **Attention enables parallelization**: Unlike RNNs (sequential), all attention scores can be computed simultaneously. This is the primary reason transformers train faster and scale better — leading to GPT, BERT, and every modern LLM.